# 06 — Ablation Study (Table 2)

Trains the three ablation rows for the report's Table 2 and compares them against the Full model (v3 at the optimal `LAMBDA_H`, produced by `04_v3.ipynb`):

- **w/o nutrition inject** — v1 MVP: keeps `g` + `L_health` but drops all architectural nutrient injection (`--mode mvp`).
- **w/o L_health** — v3 with `LAMBDA_H = 0`: keeps the hub architecture but removes the health supervision.
- **w/o flavor compound** — v3 with `--ablation_no_compound`: keeps the hub but drops the I-F / I-D flavor-compound edges.

**Prerequisite for the Full row**: run `04_v3.ipynb` first (its `outputs/v3` checkpoint is the Full reference). If it's missing the Full row is simply skipped — the ablations still print.

**Runtime**: Colab free T4 GPU, ~1.5 hr.

**How to run**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell below) if your Drive layout differs
3. (optional) Edit the **Config** cell to change `LAMBDA_H` / `TAU_PERCENTILE`
4. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# Versions follow requirements.txt; torch ships with Colab. torch_geometric
# must match the installed torch build — if its import fails later, install
# the PyG wheel for the torch version printed here:
#   https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
import torch; print('torch', torch.__version__)
!pip install -q 'torch_geometric>=2.4' 'pandas>=2.0' 'numpy>=1.24' matplotlib

## 2. Mount Drive (code + data both live in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR     = f'{PROJECT_ROOT}/code'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'CWD        = {os.getcwd()}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# Fail fast with a clear message if the dataset isn't fully prepared.
# Training needs the substitution pairs + recipes on top of the graph;
# produce them with src/convert_data.py (or notebooks/01_setup_data.ipynb).
_required = ['flavorgraph_edges.csv', 'nodes_filtered.csv', 'usda_mapping.json',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv', 'recipes.json']
_missing = [f for f in _required if not os.path.exists(f'{DATA_DIR}/{f}')]
if _missing:
    raise FileNotFoundError(
        f'Missing data files in {DATA_DIR}: {_missing}. Prepare the dataset '
        'first (src/convert_data.py or notebooks/01_setup_data.ipynb).')
print('[data] all required files present')

## 3. Config — hyper-parameters

The only knobs worth changing. `LAMBDA_H` weights `L_health` against the
GISMo substitution loss; `TAU_PERCENTILE` sets the goal-threshold percentile
(0 = "any reduction counts"). `LAMBDA_H = 1.0` is used for every model in the report.

In [ ]:
LAMBDA_H       = 1.0   # health-loss weight (same for all models)
TAU_PERCENTILE = 0     # goal threshold percentile (0 = any reduction)
print(f'LAMBDA_H = {LAMBDA_H},  TAU_PERCENTILE = {TAU_PERCENTILE}')

## 4. Smoke tests (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Set RUN_SMOKE = False to skip.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v1.py --mode mvp --lambda_h {LAMBDA_H} --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v1_mvp
    print('\n[smoke] OK')

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Set RUN_SMOKE = False to skip.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v3.py --lambda_h {LAMBDA_H} --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v3
    print('\n[smoke] OK')

## 5. Train the three ablation rows

All hold `TAU_PERCENTILE` fixed. The first two hold `LAMBDA_H` at the optimal value (only the architecture / edges change); the **w/o L_health** row sets `--lambda_h 0` by definition. All four `g` overrides are evaluated so the table can show g-sensitivity.

In [ ]:
# Ablation 1: w/o nutrition inject (v1 MVP)
!python src/train_v1.py --mode mvp --lambda_h {LAMBDA_H} --tau_percentile {TAU_PERCENTILE} \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v1mvp

In [ ]:
# Ablation 2: w/o L_health (v3, lambda=0)
!python src/train_v3.py --lambda_h 0 --tau_percentile {TAU_PERCENTILE} \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v3_lam0

In [ ]:
# Ablation 3: w/o flavor compound (v3, I-N hub kept but I-F/I-D edges dropped)
!python src/train_v3.py --lambda_h {LAMBDA_H} --tau_percentile {TAU_PERCENTILE} --ablation_no_compound \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v3_no_compound

## 6. Table 2 — ablation vs Full (+ delta-from-Full)

Reading down the delta view: a negative number means removing that component *hurts* the metric, i.e. the component helps.

In [ ]:
import os, json, sys, contextlib, io
import pandas as pd

sys.path.insert(0, f'{CODE_DIR}/src')
from evaluate_health import compute_health_metrics
from evaluate_flavor import evaluate_flavor
from evaluate_id_ood import evaluate_id_ood


def _quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def collect_metrics(pred_path):
    if not os.path.exists(pred_path):
        return None
    with open(pred_path) as f:
        pred = json.load(f)
    if 'metrics' not in pred:
        print(f'  [skip] {pred_path}: no metrics block')
        return None
    # Tolerate a missing key (older/partial file) — drop just that column.
    out = {k: pred['metrics'][k] for k in ('MRR', 'Hit@1', 'Hit@3', 'Hit@10')
           if k in pred['metrics']}
    try:
        h = _quiet(compute_health_metrics, pred_path,
                   f'{DATA_DIR}/usda_mapping.json', save_to=None)
        out['sugar_sat']   = h['pred']['sugar_g']['satisfaction_rate']
        out['sodium_sat']  = h['pred']['sodium_mg']['satisfaction_rate']
        out['d_sugar_g']   = h['pred']['sugar_g']['avg_delta']
        out['d_sodium_mg'] = h['pred']['sodium_mg']['avg_delta']
    except Exception as e:
        print(f'  [health failed] {e}')
    try:
        fv = _quiet(evaluate_flavor, pred_path,
                    f'{DATA_DIR}/flavorgraph_edges.csv', save_to=None)
        out['flavor_cos'] = fv['pred_cosine']['mean']
    except Exception as e:
        print(f'  [flavor failed] {e}')
    try:
        io_ = _quiet(evaluate_id_ood, pred_path,
                     f'{DATA_DIR}/pairs_train.csv', save_to=None)
        out['ID_MRR']  = io_['id']['MRR']
        out['OOD_MRR'] = io_['ood']['MRR']
    except Exception as e:
        print(f'  [idood failed] {e}')
    return out


def show_table(path_dict):
    rows = []
    for label, path in path_dict.items():
        m = collect_metrics(path)
        if m is None:
            print(f'  skip [{label}]: not found  {path}')
            continue
        rows.append({'config': label, **m})
    if not rows:
        print('(no predictions found)')
        return None
    df = pd.DataFrame(rows).set_index('config')
    cols = ['MRR', 'Hit@1', 'Hit@10', 'sugar_sat', 'sodium_sat',
            'flavor_cos', 'ID_MRR', 'OOD_MRR']
    print(df[[c for c in cols if c in df.columns]].round(2).to_string())
    return df

In [ ]:
print('=== Table 2: Ablation Study (g=auto) ===')
ABL = {
    'Full (v3)':                f'{OUTPUT_DIR}/v3/test_predictions_v3_auto.json',
    'w/o nutrition inject':     f'{OUTPUT_DIR}/v1mvp/test_predictions_mvp_auto.json',
    'w/o L_health (lambda=0)':  f'{OUTPUT_DIR}/v3_lam0/test_predictions_v3_auto.json',
    'w/o flavor compound':      f'{OUTPUT_DIR}/v3_no_compound/test_predictions_v3_auto.json',
}
df = show_table(ABL)

if df is not None:
    df.to_csv(f'{OUTPUT_DIR}/table2_ablation.csv')
    print(f'\n[saved] {OUTPUT_DIR}/table2_ablation.csv')
    full_key = 'Full (v3)'
    if full_key in df.index:
        dcols = [c for c in ['MRR', 'Hit@10', 'sugar_sat', 'sodium_sat', 'flavor_cos']
                 if c in df.columns]
        delta = (df[dcols] - df.loc[full_key, dcols]).round(2)
        print('\n--- delta from Full (negative = removing the component hurts) ---')
        print(delta.drop(index=full_key).to_string())